# Video Pipeline

This notebook demonstrates docling's video processing pipeline, which:
- Transcribes audio using Whisper ASR
- Samples representative frames using scene-change detection
- Optionally assigns speaker labels via diarization
- Exports to HTML, Markdown, JSON, or WebVTT

**Use cases:**
- Business meeting recordings → searchable transcript with frames
- Lecture videos → structured notes with slide captures
- Any video → subtitle file (.vtt) that can be attached back to the video


## Setup

Install docling with video support:
```bash
pip install docling
pip install openai-whisper  # ASR backend
pip install resemblyzer scikit-learn  # optional: speaker diarization
```

FFmpeg must also be installed on your system:
```bash
# macOS
brew install ffmpeg
# Ubuntu/Debian
sudo apt-get install ffmpeg
```

In [ ]:
from pathlib import Path

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import VideoPipelineOptions
from docling.document_converter import DocumentConverter, VideoFormatOption
from docling.utils.video_frame_sampling import VideoFrameSamplingMode

## Basic Usage

Convert a video using fixed-interval frame sampling (one frame every 10 seconds):

In [ ]:
VIDEO_PATH = "path/to/your/video.mp4"  # replace with your video

dc = DocumentConverter(allowed_formats=[InputFormat.VIDEO])
result = dc.convert(VIDEO_PATH)

print(f"Status: {result.status}")
print(f"Transcript segments: {len(result.document.texts)}")
print(f"Frames captured: {len(result.document.pictures)}")

## Scene-Change Detection

For meeting recordings, scene-change sampling captures frames at meaningful transitions
rather than at fixed intervals. The `prominence` parameter controls sensitivity —
lower values detect more subtle scene changes.

**Recommended settings:**
- Business meetings: `prominence=0.03`
- Lectures with slides: `cuts_per_minute=2`
- Dynamic content: `prominence=0.01`

In [ ]:
opts = VideoPipelineOptions(
    frame_sampling_mode=VideoFrameSamplingMode.SCENE_CHANGE,
    scene_change_prominence=0.03,  # recommended for meetings
    min_scene_duration_seconds=2.0,
)

dc = DocumentConverter(
    allowed_formats=[InputFormat.VIDEO],
    format_options={InputFormat.VIDEO: VideoFormatOption(pipeline_options=opts)},
)
result = dc.convert(VIDEO_PATH)

# Print transcript with timestamps
for item, _ in result.document.iterate_items():
    if hasattr(item, "text") and item.text:
        source = getattr(item, "source", None)
        ts = source[0].start_time if source else 0.0
        m, s = divmod(int(ts), 60)
        print(f"[{m:02d}:{s:02d}] {item.text.strip()}")

## Speaker Diarization

Enable speaker diarization to identify who is speaking in each segment.
Requires `resemblyzer` and `scikit-learn`.
Speaker count is automatically detected.

In [ ]:
opts = VideoPipelineOptions(
    frame_sampling_mode=VideoFrameSamplingMode.SCENE_CHANGE,
    scene_change_prominence=0.03,
    enable_diarization=True,  # requires resemblyzer
)

dc = DocumentConverter(
    allowed_formats=[InputFormat.VIDEO],
    format_options={InputFormat.VIDEO: VideoFormatOption(pipeline_options=opts)},
)
result = dc.convert(VIDEO_PATH)

for item, _ in result.document.iterate_items():
    if hasattr(item, "text") and item.text:
        source = getattr(item, "source", None)
        track = source[0] if source else None
        ts = track.start_time if track else 0.0
        speaker = track.voice if track else None
        m, s = divmod(int(ts), 60)
        spk = f" [{speaker}]" if speaker else ""
        print(f"[{m:02d}:{s:02d}]{spk} {item.text.strip()}")

## Export Formats

The `DoclingDocument` produced by the video pipeline can be exported to any format
that docling supports.

In [ ]:
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

# HTML — transcript with timestamps and embedded frames
result.document.save_as_html(output_dir / "video.html")

# Markdown — plain transcript
result.document.save_as_markdown(output_dir / "video.md")

# JSON — full structured document
result.document.save_as_json(output_dir / "video.json")

# WebVTT — subtitle file
result.document.save_as_vtt(output_dir / "video.vtt")

print("Exported to:", list(output_dir.iterdir()))

## Use Case: Video with Embedded Subtitles

Export to WebVTT and attach back to the video using FFmpeg, creating a video file
with embedded subtitles that can be played in VLC or any modern media player.

In [ ]:
import subprocess

vtt_path = output_dir / "video.vtt"
output_video = output_dir / "video_with_subtitles.mkv"

result.document.save_as_vtt(vtt_path)

subprocess.run([
    "ffmpeg", "-i", VIDEO_PATH,
    "-i", str(vtt_path),
    "-c", "copy",
    str(output_video),
    "-y",
], check=True)

print(f"Video with subtitles saved to: {output_video}")
print("Open with VLC: Subtitle → Sub Track → Track 1")